## Configuration and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer

# --- CONFIGURATION ---
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DATASET_PATH = "GBaker/MedQA-USMLE-4-options"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 5000
RANDOM_SEED = 42
SYSTEM_PROMPT = "You are a medical expert taking the USMLE board exam. Read the clinical vignette and options carefully. Output ONLY the single letter of the correct answer (A, B, C, or D). Do not provide any explanation."


## EDA

In [ ]:
def run_medqa_eda():
    print("1. Loading MedQA Dataset and Tokenizer...")
    dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)
    df = dataset.to_pandas()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    print("\n2. Analyzing Answer Distribution (Class Imbalance)...")
    answer_counts = df["answer_idx"].value_counts(normalize=True) * 100
    print("Answer Option Distribution (%):")
    print(answer_counts)

    plt.figure(figsize=(6, 4))
    sns.barplot(x=answer_counts.index, y=answer_counts.values, palette="muted")
    plt.title("Distribution of Correct Answers (A, B, C, D)")
    plt.ylabel("Percentage")
    plt.show()

    print("\n3. Calculating Token Length Distributions...")

    def calculate_medqa_lengths(row):
        question = str(row.get("question", "")).strip()
        options = row.get("options", {})

        options_list = []
        for letter in ["A", "B", "C", "D"]:
            if letter in options:
                options_list.append(f"{letter}) {options[letter]}")
        options_str = "\n".join(options_list)

        input_text = f"{question}\n\nOptions:\n{options_str}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": str(row.get("answer_idx", "")).strip()},
        ]

        full_text = tokenizer.apply_chat_template(messages, tokenize=False)
        total_tokens = len(tokenizer.encode(full_text))

        prompt_text = tokenizer.apply_chat_template(messages[:2], tokenize=False)
        prompt_tokens = len(tokenizer.encode(prompt_text))

        return pd.Series([prompt_tokens, (total_tokens - prompt_tokens), total_tokens])

    sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).copy()
    sample_df[["prompt_tokens", "completion_tokens", "total_tokens"]] = sample_df.apply(
        calculate_medqa_lengths, axis=1
    )

    print("\nToken Length Percentiles:")
    print(
        sample_df[["prompt_tokens", "completion_tokens", "total_tokens"]].describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\n4. Plotting Token Distributions...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.histplot(
        sample_df["prompt_tokens"], bins=50, ax=axes[0], color="blue"
    ).set_title("Prompt Token Length")
    sns.histplot(
        sample_df["completion_tokens"], bins=50, ax=axes[1], color="orange"
    ).set_title("Completion Token Length (~1)")
    sns.histplot(
        sample_df["total_tokens"], bins=50, ax=axes[2], color="green"
    ).set_title("Total Sequence Length")
    plt.tight_layout()
    plt.show()

    print("\n5. Medical Vocabulary Analysis (Router Prep)...")
    vectorizer = CountVectorizer(stop_words="english", max_features=25)
    word_counts = vectorizer.fit_transform(df["question"].astype(str)).sum(axis=0)
    words_freq = [
        (word, word_counts[0, idx]) for word, idx in vectorizer.vocabulary_.items()
    ]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    words, counts = zip(*words_freq)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=list(counts), y=list(words), palette="magma")
    plt.title("Top 25 Informative Words in USMLE Vignettes")
    plt.show()


## Execution

In [ ]:
if __name__ == "__main__":
    run_medqa_eda()
